# A/B Testing

Estimating causal effects using non-experimental data (ie, data that was not
gathered in a lab environment) is difficult because of selection bias and other
factors that make our estimated values differ from the true population effects.
A/B Tests (also known as Live Experiments) are experiments designed to cleanly estimate
causal effects under ideal conditions.

To illustrate A/B tests, we will replicate some of the findings from the
Student-Teacher Achievement Ratio (STAR) experiment. The goal of this experiment
was to evaluate the effect of class size on student performance.

The experiment randomly assigning students to one of three class types:

1. Regular-sized class
2. Regular-sized class with teacher's aide
3. Small class

Students were randomly assigned to each group and their performance was measured
over kindergarten, first, second and third grade using standardized reading and
math tests.

## Environment configuration

Imports

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind
import statsmodels.api as sm

Configure pandas to display all columns

In [ ]:
pd.set_option('display.max_columns', None)

## Data

Read dataset ([documentation can be found here](
    https://vincentarelbundock.github.io/Rdatasets/doc/AER/STAR.html
)).

In [ ]:
URL_DATA = 'https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/refs/heads/master/csv/AER/STAR.csv'
df = pd.read_csv(URL_DATA)

### Explore dataset

The Student Teacher Achievement Ratio (STAR) dataset is derived from an 80s
randomized experiment conducted in Tennessee. The primary goal of the experiment
was to evaluate the effect of class size on student academic achievement. It
includes variables such as student test scores, class type (small class, regular
class, or regular-sized class with a teacher's aide), and other student and
teacher characteristics.

**QUESTION**: Can you think of any biases?

In [ ]:
df.head()

### Data Cleaning and Feature Engineering

Let's drop observations where we don't know the kid's
birthdate, ethnicity or gender.

**QUESTION**: What about `lunchk`, `lunch1`, `lunch2` and `lunch3`?

In [ ]:
# Count NaNs
display(df[['gender', 'ethnicity', 'birth']].isna().sum())
display(df[['gender', 'ethnicity', 'birth']].isna().any(axis=1).sum())
round(
    df[['gender', 'ethnicity', 'birth']].isna().any(axis=1).sum() / df.shape[0] * 100,
    2
)

# Drop NaNs since they're only 1.6%
df.dropna(subset=['gender', 'ethnicity', 'birth'], inplace=True)

Convert `date` to a useful numeric dtype

In [ ]:
# Declare function to convert birth to year fraction
def quarter_to_numeric(birth_str):
    year, quarter_str = birth_str.split(' ')
    quarter_num = int(quarter_str[1])
    return float(year) + (quarter_num - 1) / 4.0

# Apply function
df['birth_numeric'] = df['birth'].apply(quarter_to_numeric)
display(df[['birth', 'birth_numeric']].sample(10))

Convert string columns to binary indicators

In [ ]:
# Shorten treatment group names
for col in ['stark', 'star1', 'star2', 'star3']:
    df[col] = df[col].str.replace('regular+aide', 'aide')

# Add constant term
df['const'] = 1

# Treatment columns to dummies
cols_str = [
    'stark', 'star1', 'star2', 'star3',
    'gender', 'ethnicity', 'birth', 'lunchk', 'lunch1', 'lunch2', 'lunch3',
    'degreek', 'degree1', 'degree2', 'degree3', 'schoolk', 'school1', 'school2',
    'school3', 'tethnicityk', 'tethnicity1', 'tethnicity2', 'tethnicity3',
    'lunchk', 'lunch1', 'lunch2', 'lunch3'
]

# Keep original dataset and concat with dummies (just in case)
df = pd.concat(
    objs=[df, pd.get_dummies(df[cols_str], dummy_na=True, drop_first=False, dtype=int)],
    axis=1
)

## Hypothesis testing

* $H_0: \mu_T - \mu_C = 0$
* $H_1: \mu_T - \mu_C \neq 0$

### Pooled *t*-test

Assuming that $H_0$ is true and that the variance of both groups (treatment and
control) is equal, the $t$-statistic is defined as follows:

$$
    t = \frac{
        \bar{Y}_T - \bar{Y}_C
    }{
        S_p \sqrt{\frac{1}{n_T} + \frac{1}{n_C}}
    }
$$

Where the pooled standard deviation ($S_p$) is calculated as:
$$
    S_p = \sqrt{\frac{
        (n_T - 1)S_T^2 + (n_C - 1)S_C^2
    }{
        n_T + n_C - 2
    }}
$$

In [ ]:
t_math3_pooled = ttest_ind(
    a=df.loc[df['star3'].eq('small'), 'math3'].values,
    b=df.loc[df['star3'].eq('regular'), 'math3'].values,
    equal_var=True,
    nan_policy='omit',
    alternative='greater'
)

### Welch's *t*-test

Assuming $H_0$ is true and that the variance is potentially different between both
groups, the $t$ statistic is defined as follows:

$$
    t = \frac{
        \bar{Y}_T - \bar{Y}_C
    }{
        \sqrt{\frac{S_T^2}{n_T} + \frac{S_C^2}{n_C}
    }}
$$

In [ ]:
t_math3_ind = ttest_ind(
    a=df.loc[df['star3'].eq('small'), 'math3'].values,
    b=df.loc[df['star3'].eq('regular'), 'math3'].values,
    equal_var=False,
    nan_policy='omit',
    alternative='greater'
)

In [ ]:
# Display ATE
print(
    df.loc[df['star3'].eq('small'), 'math3'].mean()
    - df.loc[df['star3'].eq('regular'), 'math3'].mean()
)

**QUESTION 2**: Significance aside, what's the *actual* estimated causal effect
of small class size?

It is not the value of the statistic! Instead, it's the value of $\bar{y}_{control} -
\bar{y}_{small}$.

**QUESTION 3**: How do you compare small class size VS regular + aide?

Perform a t-test for $\bar{y}_{small} - \bar{y}_{aide}$.

**QUESTION 4**: How can you control for one of the biases from Q1 using this method?

Not directly. This inconvinience justifies our leap to linear regression.

### Linear Regression

As you have probably figured out, we can answer these questions using linear regression
and it's much less painful this way (we can get the same results with less effort).

#### Naive model

To estimate the effect of being assigned to a small class size on math scores, we will
estimate the following model only on 3rd-grade students who are either in
regular or small groups.

$$
  E(Y|\text{ class type }) = \beta_0 + \beta_1 G_{\text{small}}
$$

In [ ]:
# Declare mask
mask = df['star3'].isin(['small', 'regular'])

# Fit model on masked data
lm_math3 = sm.OLS(
    endog=df.loc[mask, 'math3'],
    exog=df.loc[mask, ['const', 'star3_small']],
    missing='drop',
    #has_const=True
).fit()

In [ ]:
lm_math3.pvalues['star3_small']

We can also see the comparison between all three groups using a single model.

In [ ]:
# Fit model on masked data
lm_math3 = sm.OLS(
    endog=df.loc[mask, 'math3'],
    exog=df.loc[mask, ['const', 'star3_small', 'star3_aide']],
    missing='drop',
    has_const=True
).fit()

# Results
print(lm_math3.summary())

**QUESTION 5**: How can we compare `small` VS `regular+aide`?

In [ ]:
# Hypothesis: star3_small - star3_regular+aide = 0
comparison_result = lm_math3.t_test('star3_small - star3_aide = 0')

print(comparison_result)

The output above shows the results of the t-test comparing the 'small' class type to the 'regular+aide' class type. You can look at the `t value` and `P>|t|` to determine if the difference is statistically significant. A small p-value (typically less than 0.05) would indicate a significant difference between the two groups.

#### Balance Tests (Testing Randomness)

Before we move on to adding controls to the naive models from the
previous section, we will use balance tests to see if the treatment
actually appears to be randomly assigned.

The core idea is that if the treatment is truly randomly assigned, then
it **SHOULD NOT** work as a feature to predict any observable covariate.
Of course, we can only test on features we can observe, so we can never
really disprove that treatment status and some unobservable variable are
related.

1. Binary treatment and numeric column

    * birth ~ small/regular

In [ ]:
balance_bin_birth = ttest_ind(
    a=df.loc[df['star3'].eq('small'), 'birth_numeric'],
    b=df.loc[df['star3'].eq('regular'), 'birth_numeric'],
    equal_var=False,
    nan_policy='omit'
)
print('  * p-value:', balance_bin_birth.pvalue)

2. Binary treatment and binary covariate (gender ~ small/regular)

In [ ]:
balance_bin_gender = ttest_ind(
    a=df.loc[df['star3'].eq('small'), 'gender_male'],
    b=df.loc[df['star3'].eq('regular'), 'gender_male'],
    equal_var=True,
    nan_policy='omit'
)
print(balance_bin_gender.pvalue)

3. Multiclass treatment and numeric column

In [ ]:
# Fit the OLS model
balance_multi_birth = sm.OLS(
    endog=df['birth_numeric'],
    exog=df[['const', 'star3_small', 'star3_regular']]
).fit()
print(balance_multi_birth.f_pvalue)  # Oh no!!!

4. Multiclass treatment and multiclass covariate

In [ ]:
chi2, p_value, dof, expected = chi2_contingency(
    pd.crosstab(df['stark'], df['ethnicity'])
)
print(f"P-value: {p_value:.3f}")  # Borderline

#### Adjusted model

In [ ]:
# Declare X
cols_X = [
    'const', 'star3_small', 'star3_aide',
    'birth_numeric', 'gender_male', 'ethnicity_cauc', 'tethnicity3_cauc', 'experience3'
]

# Fit model
lm_math3_adj = sm.OLS(
    endog=df['math3'],
    exog=df[cols_X],
    missing='drop'
).fit()
print(lm_math3_adj.summary())

#### ATE on Reading Skills

Let's start off with the naive model.

In [ ]:
# Fit naive model on masked data for read3
lm_read3 = sm.OLS(
    endog=df['read3'],
    exog=df[['const', 'star3_small', 'star3_aide']],
    missing='drop',
    has_const=True
).fit()

# Results
print(lm_read3.summary())

Let's move on to a model with controls.

In [ ]:
# Declare X (same as math3 adjusted model)
cols_X = [
    'const', 'star3_small', 'star3_aide',
    'birth_numeric', 'gender_male', 'ethnicity_cauc', 'tethnicity3_cauc', 'experience3'
]

# Fit adjusted model for read3
lm_read3_adj = sm.OLS(
    endog=df['read3'],
    exog=df[cols_X],
    missing='drop'
).fit()
print(lm_read3_adj.summary())

## Same Treatment Status

Let's analyze what happens to the estimated treatment effects when we filter for
students who always belong to the same treatment group.

In [ ]:
# Declare incremental masks
mask_2years = (df['stark'] == df['star1'])
mask_3years = mask_2years & (df['stark'] == df['star2'])
mask_4years = mask_3years & (df['stark'] == df['star3'])

## Re-running Balance Tests on the consistently grouped population (using `mask_4years`)

### Multiclass treatment and numeric column (`birth_numeric` ~ `stark`)

In [ ]:
balance_multi_birth_consistent = sm.OLS(
    endog=df.loc[mask_4years, 'birth_numeric'],
    exog=df.loc[mask_4years, ['const', 'stark_small', 'stark_aide']]
).fit()
print(f"  * F-statistic p-value (birth_numeric vs stark): {balance_multi_birth_consistent.f_pvalue:.3f}")

### Multiclass treatment and binary covariate (`gender_male` ~ `stark`)

In [ ]:
balance_multi_gender_consistent = sm.OLS(
    endog=df.loc[mask_4years, 'gender_male'],
    exog=df.loc[mask_4years, ['const', 'stark_small', 'stark_aide']]
).fit()
print(f"  * F-statistic p-value (gender_male vs stark): {balance_multi_gender_consistent.f_pvalue:.3f}")

### Multiclass treatment and multiclass covariate (`ethnicity` ~ `stark`)

In [ ]:
chi2_consistent, p_value_consistent, dof_consistent, expected_consistent = chi2_contingency(
    pd.crosstab(df.loc[mask_4years, 'stark'], df.loc[mask_4years, 'ethnicity'])
)
print(f"  * P-value (ethnicity vs stark): {p_value_consistent:.3f}")

In [ ]:
cols_X

In [ ]:
m_4years = sm.OLS(
    endog=df.loc[mask_4years, 'math3'],
    exog=df.loc[mask_4years, cols_X],
    missing='drop'
).fit()

print(m_4years.summary())